*DEMO*

This sample parses an EML file, performs redaction on all attachments and reassembles a redacted email.

In [ ]:
import requests
import base64
import os
import dotenv
import textwrap
from bs4 import BeautifulSoup
from email import policy
from email.parser import BytesParser

# Use to load the API KEY for authentication
dotenv.load_dotenv()

#url = "http://localhost:8080/"
url = "https://api.private-ai.com/community/v4/"
headers = {"Content-Type": "application/json", "x-api-key": os.environ["PRIVATEAI_API_KEY"]}

#region PAI API CALLS 
def makePaiCall_text(data):
    request = {
        "text": data,
        "link_batch":True
    }
    
    ###----------------------------------------------------------------------------###
    ### PRIVATE AI API CALL
    response = requests.post(f"{url}process/text", json=request, headers=headers)
    response.raise_for_status()
    data = response.json()
    ### PRIVATE AI API CALL
    ###----------------------------------------------------------------------------###

    return data

def makePaiCall_file(data, content_type):
    request = {
        "file": { 
            "data": data,
            "content_type": content_type        
        }
    }
    
    ###----------------------------------------------------------------------------###
    ### PRIVATE AI API CALL
    response = requests.post(f"{url}process/files/base64", json=request, headers=headers)
    response.raise_for_status()
    data = response.json()
    ### PRIVATE AI API CALL
    ###----------------------------------------------------------------------------###

    return data

#endregion PAI API CALLS 

#region PARSE / Redact / rebuild EML file
#Parse out the EML file and store pertinent info for redaction in a dict to simplify the parsing and downstream usage of the eml metadata
#Due to the nature of EML and various clients the file needs to be traversed in a somewhat nonlinear format. 
#Because various nodes can be nested and moved around without impacting the overall parsing of the file.
def parse_eml(file_path):
    with open(file_path, 'rb') as f:
        msg = BytesParser(policy=policy.default).parse(f)

    parsed = {
        "from": msg.get("From"),
        "to": msg.get("To"),
        "cc": msg.get("CC"),
        "bcc": msg.get("BCC"),
        "subject": msg.get("Subject"),
        "date": msg.get("Date"),
        "body_text": None,
        "body_html": None,
        "attachments": [],
        "inline_images": []
    }

    #This section is looking for inline and attached files. Content like images can be 
    #embedded directly within the message or added as an attachment.
    def walk_parts(message):
        if message.is_multipart():
            ctype = message.get_content_type()

            if ctype == "multipart/alternative":
                # Walk each alternative part
                for part in message.iter_parts():
                    if part.get_content_type() == "text/plain":
                        parsed["body_text"] = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    elif part.get_content_type() == "text/html":
                        parsed["body_html"] = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    elif part.get_content_type() == "multipart/related":
                        # Handle nested multipart/related inside alternative
                        handle_related(part)

            elif ctype == "multipart/related":
                # Top-level related
                handle_related(message)

            else:
                # Generic multipart (mixed, etc.)
                for part in message.iter_parts():
                    walk_parts(part)

        else:
            ctype = message.get_content_type()
            if message.get_content_disposition() == "attachment":
                filename = message.get_filename()
                payload = message.get_payload(decode=True)
                b64_str = base64.b64encode(payload).decode("utf-8")                

                parsed["attachments"].append({
                    "filename": filename,
                    "content_type": ctype,
                    "data": b64_str
                })
            elif ctype == "text/plain":
                parsed["body_text"] = message.get_payload(decode=True).decode("utf-8", errors="ignore")
            elif ctype == "text/html":
                parsed["body_html"] = message.get_payload(decode=True).decode("utf-8", errors="ignore")

    def handle_related(related_msg):
        for part in related_msg.iter_parts():
            if part.get_content_type() == "text/html":
                parsed["body_html"] = part.get_payload(decode=True).decode("utf-8", errors="ignore")

            elif part.get_content_disposition() == "inline":
                cid = part.get("Content-ID")
                filename = part.get_filename()

                # Safely decode payload
                payload_bytes = part.get_payload(decode=True)
                if payload_bytes is None:
                    # Fallback: decode raw base64 string
                    raw_payload = part.get_payload()
                    try:
                        payload_bytes = base64.b64decode(raw_payload)
                    except Exception:
                        payload_bytes = b""

                b64_str = base64.b64encode(payload_bytes).decode("utf-8")
                
                parsed["inline_images"].append({
                    "cid": cid.strip("<>") if cid else None,
                    "filename": filename,
                    "content_type": part.get_content_type(),
                    "data": b64_str
                })

            else:
                walk_parts(part)


    walk_parts(msg)
    return parsed

def restore_original_cids(original_html, sanitized_html, inline_images):
    """
    Replace <img src="cid:..."> in sanitized_html with the original CIDs
    from parsed_email.inline_images.
    """
    if not sanitized_html or not inline_images:
        return sanitized_html

    soup = BeautifulSoup(sanitized_html, "html.parser")

    for img in soup.find_all("img"):
        src = img.get("src", "")
        if src.startswith("cid:"):
            # Find matching original image by filename or cid
            for orig in inline_images:
                if orig.get("cid"):
                    img["src"] = f"cid:{orig['cid']}"
                    break

    return str(soup)
    
def send_to_api(parsed_email):
    # === Call 1: Send core email metadata (headers + body) ===    
    #Simplify the PAI call by sending each component of the message as a link batch. Low context around names / subjects but higher context within the html / text body.
    text_payload = [
        parsed_email.get("from"),
        parsed_email.get("to") or '', #return an empty string if optional values are None
        parsed_email.get("cc") or '', #return an empty string if optional values are None
        #parsed_email.get("bcc"),
        parsed_email.get("subject"),
        parsed_email.get("date"),
        parsed_email.get("body_text"),
        parsed_email.get("body_html")  # send entire HTML, not just <body>
    ]
    core_json = makePaiCall_text(text_payload)

    response_from = core_json[0]["processed_text"]
    response_to = core_json[1]["processed_text"]
    response_cc = core_json[2]["processed_text"]
    response_subject = core_json[3]["processed_text"]
    response_date = core_json[4]["processed_text"]
    response_body_text = core_json[5]["processed_text"]
    response_body_html_sanitized = core_json[6]["processed_text"]

    # Overwrite sanitized HTML with original CIDs
    # In the event that inline image CIDs are redacted this would break the original message. As such we retain the original CIDs and re-insert them
    #An alternative to this would be to convert the inline attachments to normal attachments.
    response_body_html = restore_original_cids(
        parsed_email.get("body_html"),
        response_body_html_sanitized,
        parsed_email.get("inline_images", [])
    )

    # === Call 2: Loop through attachments ===
    attachment_responses = []
    for att in parsed_email.get("attachments", []):
        resp_json = makePaiCall_file(att["data"], att["content_type"])

        processed_file = resp_json['processed_file']
        # Rebuild attachment entry with filename preserved
        attachment_responses.append({
            "filename": att.get("filename"),
            "content_type": att.get("content_type"),
            "data": processed_file
        })

    # === Call 3: Loop through inline images ===
    inline_responses = []
    for img in parsed_email.get("inline_images", []):
        resp_json = makePaiCall_file(img["data"], img["content_type"])

        processed_file = resp_json['processed_file']
        # Rebuild inline image entry with cid + filename preserved
        inline_responses.append({
            "cid": img.get("cid"),
            "filename": img.get("filename"),
            "content_type": img.get("content_type"),
            "data": processed_file
        })

    # Return unified schema
    return {
        "from":response_from,
        "to": response_to,
        "cc": response_cc,
        "subject": response_subject,
        "date": response_date,
        "body_text": response_body_text,
        "body_html": response_body_html,
        "attachments": attachment_responses,
        "inline_images": inline_responses
    }

#Reconstruct a new EML using the original as a template. Basically update the original in order to preserve CIDs and original formatting as much as possible.
def reconstruct_eml(original_path, api_response, output_path):
    # Parse the original message again
    with open(original_path, 'rb') as f:
        msg = BytesParser(policy=policy.default).parse(f)

    # Update headers
    if api_response.get("from"):
        msg.replace_header("From", api_response["from"])
    if api_response.get("to"):
        msg.replace_header("To", api_response["to"])
    if api_response.get("cc"):
        msg.replace_header("CC", api_response["cc"])
    if api_response.get("bcc"):
        msg.replace_header("BCC", api_response["bcc"])
    if api_response.get("subject"):
        msg.replace_header("Subject", api_response["subject"])
    if api_response.get("date"):
        msg.replace_header("Date", api_response["date"])

    # Walk through parts and update body/attachments/inline images
    def update_parts(message):
        if message.is_multipart():
            for part in message.iter_parts():
                update_parts(part)
        else:
            ctype = message.get_content_type()
            disp = message.get_content_disposition()

            # Update plain text body
            if ctype == "text/plain" and api_response.get("body_text") is not None:
                message.set_payload(api_response["body_text"], charset="utf-8")

            # Update HTML body
            elif ctype == "text/html" and api_response.get("body_html") is not None:
                message.set_payload(api_response["body_html"], charset="utf-8")

            # Update attachments
            elif disp == "attachment":
                filename = message.get_filename()
                for att in api_response.get("attachments", []):
                    if att.get("filename") == filename:
                        data = base64.b64decode(att["data"])
                        # Encode with 76‑char line wrapping - this magic number is from the mime spec
                        # Line length wrapping is critical as most mail clients will not properly render inline images / attachments if they are not wrapped at 76
                        b64_wrapped = "\r\n".join(textwrap.wrap(base64.b64encode(data).decode("utf-8"), 76))
                        message.set_payload(b64_wrapped)
                        message.set_type(att["content_type"])
                        message.set_param("name", filename)
                        break
            
            # Update inline images
            elif disp == "inline":
                cid = message.get("Content-ID")
                filename = message.get_filename()
                # Find matching inline image in API response
                for img in api_response.get("inline_images", []):
                    if (cid and img.get("cid") == cid.strip("<>")) or img.get("filename") == filename:
                        data = base64.b64decode(img["data"])
                         # Encode with 76‑char line wrapping
                         # Line length wrapping is critical as most mail clients will not properly render inline images / attachments if they are not wrapped at 76
                        b64_wrapped = "\r\n".join(textwrap.wrap(base64.b64encode(data).decode("utf-8"), 76))
                        message.set_payload(b64_wrapped)
                        message.set_type(img["content_type"])
                        message.set_param("name", filename)
                        # Ensure Content-ID is preserved
                        if cid:
                            message.replace_header("Content-ID", f"<{img['cid']}>")
                        break

    update_parts(msg)

    # Write updated message back to disk
    with open(output_path, "wb") as f:
        f.write(msg.as_bytes())

    print(f"Reconstructed EML saved to {output_path}")

#endregion PARSE / Redact / rebuild EML file

# === MAIN EXECUTION ===
if __name__ == "__main__":
    # Parse original EML
    test_file = '.\\data\\sample_PCI_email.eml'
    parsed = parse_eml(test_file)

    # Send to REST API
    api_response = send_to_api(parsed)

    # Reconstruct EML from API response
    reconstruct_eml(test_file, api_response, test_file + '.redacted.eml')

Reconstructed EML saved to .\data\sample_PCI_email.eml.redacted.eml
